<div style="background-color:#5A3516; color:#F3EEE6; padding:22px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h1 style="color:#F3EEE6; margin-bottom:0;"><b>Machine Learning II — Customer Segmentation</b></h1>
<h3 style="color:#D8C0B4; margin-top:6px;">Notebook 3 — Clustering Modelling and Validation</h3>
</div>

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Workflow overview.</b> This notebook develops the segmentation as a sequence of modelling decisions rather than as a single KMeans run. Alternative feature spaces, scalers and clustering families are compared first; the final solution is then validated through silhouette analysis, hierarchical benchmarks, based on density checks, consensus stability and business profiling. The final labels are interpreted using the full customer profile, while the distance calculation is kept focused on the variables that best separate purchasing behaviour.
</div>


<div style="background-color:#5A3516; color:#F3EEE6; padding:18px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin-top:0;"><b>Index</b></h2>
<ol>
<li>Imports and data loading</li>
<li>Candidate feature sets and scalers</li>
<li>Diagnostics A — scaler comparison, elbow and dendrograms</li>
<li>Diagnostics B — feature set × k silhouette grid</li>
<li>Model configuration</li>
<li>Model fitting and validation</li>
<li>Method benchmarks before ensemble</li>
<li>Ensemble / consensus clustering</li>
<li>Segment profiling</li>
<li>Reattach outliers and export</li>
<li>Final modelling conclusion</li>
</ol>
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1) Imports and data loading</b></h2>
</div>

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

import utils_clustering as uc

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

In [ ]:
DATA_DIR = Path("../datasets")

regular = pd.read_csv(DATA_DIR / "info_clustering_unscaled.csv", index_col="customer_id")
outliers = pd.read_csv(DATA_DIR / "outlier_dataset.csv", index_col="customer_id")

print("Regular:", regular.shape, "| Outliers:", outliers.shape,
      "| Total:", len(regular) + len(outliers))
regular.head()

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2) Candidate feature sets and scalers</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">This section defines the modelling search space. The comparison separates absolute spend, transformed spend, promotion behaviour and optional demographic or engagement variables, so that the final feature set can be justified with evidence rather than selected manually.</p>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
Why some columns repeat information: absolute <code>lifetime_spend_*</code> and <code>log_total_spend</code> both describe spend intensity; <code>tenure</code> = current year − <code>year_first_transaction</code>; <code>total_children</code> = <code>kids_home + teens_home</code>; <code>typical_hour</code> is replaced by its cyclic <code>sin/cos</code>. Avoid putting duplicated views of the same thing in one feature set. Identity/geo (<code>customer_id, latitude, longitude, is_male, customer_loyalty_flag</code>) are best left out of the distance and used for profiling.
</div>

In [ ]:
candidate_sets = uc.build_candidate_feature_sets(regular)
for name, (cols, logabs) in candidate_sets.items():
    print(f'{name:32s} | {len(cols):2d} cols | log_abs_spend={logabs}')

scaler_options = ['Standard', 'MinMax', 'Robust', 'None']
print('\nScalers available:', scaler_options)

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>3) Diagnostics A — scaler comparison, elbow and dendrograms</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">This block checks whether the selected structure depends strongly on preprocessing choices. Scaler comparison evaluates distance sensitivity, the elbow checks compactness gains, and the dendrograms inspect whether hierarchical structure is visible under several linkage rules.</p>
</div>


### 3.1 — Scaler comparison (silhouette vs k)

In [ ]:
INSPECT_SET = "spend + promo no groceries"  
INSPECT_SCALER = "MinMax"  

inspect_cols, inspect_logabs = candidate_sets[INSPECT_SET]
print(f'Inspecting: {INSPECT_SET}  ({len(inspect_cols)} cols, log_abs={inspect_logabs})')
print(f'Inspection scaler: {INSPECT_SCALER}')


In [ ]:
scaler_scores = uc.compare_scalers(regular, inspect_cols, k_range=range(2, 14))
uc.plot_scaler_comparison(scaler_scores)
scaler_scores.pivot(index='k', columns='scaler', values='silhouette')

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Scaler selection interpretation.</b><br><br>
Both RobustScaler and MinMaxScaler were considered because the spend variables are skewed and measured on different scales. RobustScaler is usually safer when extreme observations remain in the modelling set, since it reduces the influence of unusually high values. However, in this project the most extreme customers are already held aside as outliers before fitting the final centroids. Under this cleaner modelling set, MinMaxScaler becomes a strong option because it places all clustering variables on the same 0-1 range while preserving their relative ordering.<br><br>
For the selected feature set and final <i>k = 8</i> configuration, MinMaxScaler kept the clusters cleaner and more interpretable than the alternatives tested. The resulting clusters still correspond to clear business groups. Therefore, MinMaxScaler is selected for the final model because it improves geometric separation without sacrificing interpretability.
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>A note on groceries.</b> Groceries are kept in the dataset for profiling because they are behaviourally important. However, groceries can dominate the distance calculation because most customers spend heavily there. For this reason, the candidate grid includes versions with and without <code>lifetime_spend_groceries</code>. If the without groceries option gives clearer personas, use it for KMeans and still use groceries afterwards to describe the segments.
</div>


### 3.2 — Inertia elbow

In [ ]:
X_inspect = uc.apply_feature_pipeline(regular, inspect_cols, inspect_logabs,
                                      uc.get_scaler(INSPECT_SCALER), fit=True)
k_values, inertia = uc.kmeans_elbow(X_inspect, range(1, 14))
uc.plot_elbow(k_values, inertia)

### 3.3 — Ward dendrogram (on a sample)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
Agglomerative clustering is O(n²) in memory, so we draw the dendrogram on a random sample. Reading where long vertical links get cut suggests candidate values of k.
</div>

In [ ]:
DENDRO_CUT_HEIGHT = 6.4

uc.plot_sample_dendrogram(
    X_inspect,
    title=f"Ward dendrogram (sample) - {INSPECT_SET}",
    linkage="ward",
    sample_size=3000,
    cut_height=DENDRO_CUT_HEIGHT,
    random_state=0,
)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Ward dendrogram interpretation.</b><br><br>
The Ward dendrogram is used as a visual diagnostic for compact, variance minimising group structure. Long vertical jumps indicate that relatively distant groups are being merged, so possible cut points should be considered just before those large jumps. In this dataset, the dendrogram suggests that there is meaningful hierarchical structure, but it does not provide a single unambiguous cut. Several values of <i>k</i> remain plausible, which is why the final number of clusters is not chosen from the dendrogram alone.<br><br>
Ward is nevertheless useful because it is conceptually aligned with KMeans: both favour compact groups under Euclidean geometry. Therefore, if Ward shows a broadly coherent structure, it supports the use of a based on centroids final model.
</div>


### 3.4 — Complete, average and single linkage dendrograms

Ward is usually the most compatible linkage with KMeans because both favour compact groups. Complete, average and single linkage are added as sensitivity checks: complete tends to form tighter clusters, average provides an intermediate linkage criterion, and single can reveal chaining/noise effects.


In [ ]:
ALT_DENDRO_CUT_HEIGHTS = {
    "complete": 1.2,
    "average": None,
    "single": None,
}

uc.plot_alternative_dendrograms(
    X_inspect,
    title_suffix=INSPECT_SET,
    cut_heights=ALT_DENDRO_CUT_HEIGHTS,
    sample_size=3000,
    random_state=0,
)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Linkage comparison interpretation.</b><br><br>
The complete, average and single dendrograms are interpreted as sensitivity checks rather than competing final models.
<br><br>
<b>Complete linkage</b> tends to form compact and conservative clusters because it merges groups according to their farthest pairwise distance. If complete linkage shows large jumps, it suggests that some compact groups are clearly separated from the rest of the data.
<br><br>
<b>Average linkage</b> is less strict and merges clusters based on average distances between observations. It is useful as an intermediate check: when average linkage broadly agrees with Ward or complete linkage, the hierarchical structure is less dependent on one specific linkage rule.
<br><br>
<b>Single linkage</b> is the most sensitive to chaining effects, because clusters can be joined through narrow bridges of similar observations. If single linkage produces elongated chains or less clear cuts, this does not necessarily invalidate the segmentation; it indicates that some customers lie on behavioural gradients rather than in perfectly isolated groups.
<br><br>
Overall, the dendrograms support the idea that the data contains structure, but the cut point remains partly subjective. For this reason, the final solution is selected using the dendrograms together with silhouette, scaler comparison, ensemble stability and profiling interpretability.
</div>


<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>4) Diagnostics B — feature set × k silhouette grid</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">After selecting the scaler and inspecting hierarchical structure, the silhouette grid is used to compare candidate feature spaces across several values of <i>k</i>. This provides a broader check that the final configuration is not chosen from a single diagnostic only.</p>
</div>


In [ ]:
grid = uc.silhouette_grid(regular, candidate_sets, k_range=range(6, 11),
                          scaler_name='MinMax')
grid_pivot = uc.plot_silhouette_grid(grid, title='Silhouette grid (scaler = MinMax)')
grid_pivot

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Interpretation.</b> Higher silhouette values indicate more compact and separated clusters for each candidate feature space. However, model selection also considers scaler behaviour, dendrogram structure, cluster size balance and business interpretability, since a mathematically strong solution is not necessarily the most useful segmentation.
</div>

<div style="background-color:#8F4A34; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #B2543D;">
<h2 style="color:#F3EEE6; margin:0;"><b>5) Model configuration</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">The final configuration is chosen after the diagnostic stage. The objective is not to maximise one metric in isolation, but to select a solution that is quantitatively acceptable, stable and interpretable from a customer segmentation perspective.</p>
</div>


In [ ]:
FEATURE_SET = "spend + promo no groceries"    
SCALER      = "MinMax"  
K           = 8  

In [ ]:
distance_cols, LOGABS = candidate_sets[FEATURE_SET]
profiling_cols = uc.get_profiling_features(regular, distance_cols)
scaler = uc.get_scaler(SCALER)
X = uc.apply_feature_pipeline(regular, distance_cols, LOGABS, scaler, fit=True)

print(f'Feature set : {FEATURE_SET}  ({len(distance_cols)} cols, log_abs={LOGABS})')
print(f'Scaler      : {SCALER}')
print(f'k           : {K}')
print(f'Matrix      : {X.shape}')

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Decision log.</b><br><br>
The final model uses <i>spend + promo no groceries</i>, <i>MinMaxScaler</i> and <i>k = 8</i>.<br><br>
Groceries were left out of the clustering distance because they dominated several candidate solutions and made some groups look too similar. They are still used later in profiling, since grocery spend is important to understand customer value and regular household shopping.<br><br>
MinMax scaling was kept because the most extreme customers had already been separated before modelling, and the scaled variables became easier to compare. The promotion percentage was also kept in the same scaled matrix; as it is already bounded between 0 and 1, MinMax scaling preserves its interpretation while keeping it comparable with the spend variables.<br><br>
Solutions between 6 and 10 clusters were inspected. The seven cluster option was competitive, but it merged behaviours that were clearer when eight clusters were retained. The final choice is therefore based on the combined evidence from the silhouette results, UMAP projection, stability checks and the ability to assign distinct business profiles.
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>6) Model fitting and validation</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">KMeans is fitted using the selected configuration. The solution is then validated through cluster size distribution, silhouette analysis and two dimensional PCA/UMAP projections.</p>
</div>

In [ ]:
kmeans = uc.fit_kmeans(X, K)
regular['cluster'] = kmeans.labels_
uc.plot_cluster_sizes(regular, 'cluster')
uc.cluster_sizes(regular, 'cluster')

### 6.1 — Silhouette plot

In [ ]:
avg_sil = uc.plot_silhouette_blades(X, kmeans.labels_, title=f'Silhouette — KMeans (k={K})')
print('Average silhouette:', round(avg_sil, 3))

### 6.2 — PCA and UMAP projections

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
PCA and UMAP are used as two dimensional projections for visual inspection. A subsample is used to keep the visualisation computationally efficient.
</div>

In [ ]:
X_emb, lab_emb = uc.subsample(X, kmeans.labels_, n=8000)
pca_emb = uc.embed_pca(X_emb)
uc.plot_embedding(pca_emb, lab_emb, title='PCA - KMeans Clusters', method_name='PCA')

In [ ]:
umap_emb, method = uc.embed_umap(X_emb)
uc.plot_embedding(umap_emb, lab_emb, title='UMAP - KMeans Clusters', method_name=method)

### 6.3 — Embedded feature importance check (validates the feature set)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The correlation matrix in Notebook&nbsp;1 is a <b>filter</b> method: it only removes <i>redundant</i> features and cannot say which features actually <i>drive</i> the segments. Here we add an <b>embedded</b> method &mdash; an L1-penalised (Lasso) logistic model and a Random Forest &mdash; using the cluster label as a proxy target. The L1 penalty shrinks uninformative coefficients to exactly zero (an embedded selection), and both models rank how much each feature contributes to telling the segments apart. A peaked, sparse ranking means the segmentation rests on a few interpretable drivers. This is a after the model validation of the feature set chosen in Section&nbsp;5, not a selection before modelling step.
</div>

In [ ]:
emb_importance = uc.embedded_feature_importance(
    X, kmeans.labels_, distance_cols, method='both', random_state=0
)
display(emb_importance)
uc.plot_embedded_importance(emb_importance,
                            title=f'Embedded feature importance - {FEATURE_SET}')

print('Embedded shortlist (forest importance > median):',
      uc.select_features_embedded(emb_importance, 'forest_importance'))

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>7) Method benchmarks before ensemble</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">The selected KMeans solution is compared with alternative clustering assumptions before the stability step. These checks are not meant to replace the final model automatically. They are used to verify whether the same customer structure remains plausible under hierarchical, density based and topology based views.</p>
</div>

### 7.0 — Independent method search

Each clustering family is first evaluated on its own candidate range. This avoids assuming that every method should return the same number of clusters. The comparison is used to understand the trade off between separation, cluster balance and interpretability before selecting the final operational solution.

In [ ]:
K_SEARCH_RANGE = range(2, 11)

method_benchmarks = uc.run_method_benchmarks(
    X,
    k_range=K_SEARCH_RANGE,
    random_state=0,
)

uc.display_method_benchmarks(method_benchmarks)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Benchmark interpretation.</b><br><br>
The benchmark table is intentionally separated by method family, because each algorithm reports a different set of diagnostics. KMeans is evaluated mainly through silhouette and cluster balance; hierarchical clustering adds a variance explained measure; centroid Ward uses an intermediate set of KMeans micro clusters. Keeping the tables separate avoids empty columns and makes the comparison easier to read.
<br><br>
The final model is not selected by the highest silhouette alone. A solution is only retained if it is also stable, visually coherent in the UMAP projection, reasonably balanced in size and interpretable when compared with the full customer profile.
</div>

In [ ]:
ward_sample_idx, ward_labels = uc.fit_hierarchical_sample(
    X,
    K,
    sample_size=5000,
    linkage="ward",
    random_state=0,
)
display(uc.compare_solutions(
    kmeans.labels_[ward_sample_idx],
    ward_labels,
    name_a="KMeans",
    name_b="Ward",
))

### 7.1 — Ward vs complete vs average vs single sample comparison

This fits agglomerative clustering on the same sample with four linkages. It does not replace the final KMeans model; it checks whether the broad structure is stable under different hierarchical assumptions.


In [ ]:
hierarchical_comparison, hierarchical_solution_tables = uc.compare_hierarchical_linkages(
    X,
    K,
    kmeans.labels_,
    sample_size=5000,
    random_state=0,
)

display(hierarchical_comparison)

for linkage_name, comparison_table in hierarchical_solution_tables.items():
    print(f"KMeans vs Agglomerative-{linkage_name}")
    display(comparison_table)

### 7.1.1 — Tracking benchmark labels

The benchmark labels are tracked only when they are needed for comparison, profiling or export. Direct agglomerative clustering is evaluated on a sample because full hierarchical clustering is computationally expensive for this dataset and is used here as a sensitivity check rather than as the final assignment method.


In [ ]:
label_tracking = uc.benchmark_label_tracking()
display(label_tracking)

### 7.2 — Hierarchical R2 comparison

The R2-like metric compares how much of the total variance is explained by the cluster separation for each hierarchical linkage and number of clusters. It is used as an additional diagnostic alongside silhouette and dendrogram inspection.


In [ ]:
hierarchical_r2 = uc.hierarchical_r2_grid(
    X,
    k_range=range(6, 11),
    linkages=["ward", "complete", "average", "single"],
    sample_size=5000,
    random_state=0,
)

uc.plot_hierarchical_r2(hierarchical_r2)
display(hierarchical_r2.pivot(index="k", columns="linkage", values="r2"))


### 7.3 — Hierarchical on KMeans centroids
This benchmark uses a two stage strategy. First, <b>20 KMeans small clusters</b> are created to summarise the detailed structure of the customer space. Second, Ward hierarchical clustering groups those centroids into the final larger segments. Keeping <code>MICRO_K = 20</code> makes the intermediate representation granular enough to reveal local structure, while still small enough to inspect with a dendrogram.


In [ ]:
MICRO_K = 20
micro_kmeans, centroids_df, Z = uc.apply_centroid_ward_macro(
    regular,
    X,
    distance_cols,
    K,
    micro_k=MICRO_K,
    random_state=0,
)

In [ ]:
macro_comparison = uc.compare_solutions(
    regular["cluster"].values,
    regular["macro_cluster"].values,
    name_a="Final KMeans",
    name_b="Centroid Ward macro",
)
display(macro_comparison)

uc.plot_silhouette_blades(
    X,
    regular["macro_cluster"].values,
    title=f"Silhouette - Hierarchical on KMeans centroids (k={K})",
)

In [ ]:
uc.plot_macro_embeddings(
    X,
    regular["macro_cluster"].values,
    sample_size=8000,
    random_state=0,
)

### 7.4 — DBSCAN density benchmark

DBSCAN is used as a density based benchmark. In this dataset it is useful to test whether there are dense pockets of customers, but it is not expected to be the final segmentation method because high dimensional spend data often leads to either one large cluster or a high proportion of noise points. When DBSCAN returns fewer than two non noise clusters, a silhouette value is not mathematically defined, so those parameter settings are excluded from the clean results table.

In [ ]:
dbscan_results, valid_dbscan, invalid_dbscan = uc.dbscan_benchmark_table(
    X,
    eps_values=[0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5, 2.0],
    min_samples_values=[5, 10, 20, 40],
    sample_size=8000,
    random_state=0,
)

if valid_dbscan.empty:
    print("No DBSCAN setting produced enough non-noise clusters for silhouette.")
else:
    display(valid_dbscan.head(10))

print("Parameter settings with one cluster or no valid silhouette")
display(invalid_dbscan.head(10))

In [ ]:
RUN_DBSCAN = False
DBSCAN_EPS = 0.9
DBSCAN_MIN_SAMPLES = 10

dbscan_labels = uc.maybe_fit_dbscan(
    X,
    run=RUN_DBSCAN,
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES,
)

if dbscan_labels is not None:
    regular["dbscan_cluster"] = dbscan_labels

### 7.5 — SOM Map feature map analysis

A SOM Map is used as an exploratory validation tool rather than as the final clustering algorithm. A medium sized grid is used so that neighbourhood patterns remain visible without creating hundreds of almost empty units. The hit map shows where customers concentrate, the U Matrix highlights stronger separations between neighbouring units, and the feature maps show how the main variables vary across the same customer space.

In [ ]:
SOM_GRID = (12, 12)
SOM_ITERATIONS = 1000

som_cols, X_som, som_curve, som_model = uc.run_som_diagnostic(
    regular,
    scaler_name=SCALER,
    grid=SOM_GRID,
    iterations=SOM_ITERATIONS,
    sample_size=12000,
    random_state=0,
)

In [ ]:
regular["som_unit"] = uc.assign_and_plot_som(
    regular,
    som_model,
    X_som,
    som_cols,
    grid=SOM_GRID,
)

### 7.5.1 — SOM interpretation

The SOM is used as visual support for the final segmentation. It helps check whether related variables occupy coherent regions of the same customer map.

The main aspects inspected are vegetables and hygiene for the wellness profile, technology for the technology oriented segment, promotion usage for the discount oriented segment, household variables for the family segment and complaints for service sensitivity.

Because the SOM does not optimise the same objective as KMeans, it is not used to assign final labels. It remains an exploratory diagnostic that supports the profiling stage.

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>8) Ensemble / consensus clustering</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">After comparing the individual clustering methods, a consensus KMeans procedure is used to measure stability. The same configuration is fitted several times with different random states; each customer receives the label that appears most consistently. High agreement between the single run and the consensus solution supports the final model as stable rather than accidental.</p>
</div>

In [ ]:
consensus_labels, stability = uc.consensus_report(
    X,
    kmeans.labels_,
    K,
    n_runs=25,
    random_state=0,
)
regular["cluster_stability"] = stability

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>9) Segment profiling</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">Cluster interpretation is performed after fitting the model. The profiling tables compare each segment with the overall customer base using spend, demographic, engagement, location and complaint variables. This avoids naming clusters only from the distance variables and produces more distinctive business labels.</p>
</div>


In [ ]:
spend_cols = [c for c in regular.columns if c.startswith('lifetime_spend_')]
spend_profile = uc.profile_clusters(regular, 'cluster', spend_cols)
uc.plot_profile_heatmap(spend_profile, 'Absolute spend profile by cluster')
spend_profile


### 9.1 — Petfood signal check

Petfood spending is checked separately because one previous naming option considered a focused on petfood segment. A segment should only be named after petfood if the category is clearly distinctive, either in absolute spend or as a relevant share of its total basket.


In [ ]:
if "lifetime_spend_petfood" in spend_profile.columns:
    pet_check = spend_profile.drop(index="OVERALL", errors="ignore").copy()
    pet_spend_cols = [c for c in pet_check.columns if c.startswith("lifetime_spend_")]
    pet_check["petfood_share_%"] = (
        pet_check["lifetime_spend_petfood"] / pet_check[pet_spend_cols].sum(axis=1) * 100
    )
    pet_check = pet_check[["lifetime_spend_petfood", "petfood_share_%"]].round(2)
    display(pet_check.sort_values("lifetime_spend_petfood", ascending=False))

    print("Highest absolute petfood cluster:", pet_check["lifetime_spend_petfood"].idxmax())
    print("Highest petfood share cluster:", pet_check["petfood_share_%"].idxmax())


In [ ]:
key_cols = [c for c in [
    "log_total_spend", "percentage_of_products_bought_promotion",
    "distinct_stores_visited", "lifetime_total_distinct_products",
    "customer_age", "education_level", "tenure", "total_children",
    "number_complaints",
] if c in regular.columns]

mixed_profile = uc.profile_clusters(regular, "cluster", key_cols)
uc.plot_profile_heatmap_z(mixed_profile,
                          title="Segment profile (standardised per feature)")

### 9.2 — Segment separation plots

The following plots summarise the same profiling logic in a more visual way. The scaled mean comparison highlights which clusters are highest or lowest on each variable, while the boxplots show whether the separation is concentrated around the median or driven mainly by extreme customers.


In [ ]:
scaled_profile = uc.plot_segment_separation(
    regular,
    label_col="cluster",
)

### 9.3 — Complaint behaviour by segment

Complaint behaviour is reviewed separately because it can support the interpretation of segments with higher service sensitivity or dissatisfaction risk.


In [ ]:
complaints_profile = uc.complaint_summary(
    regular,
    label_col="cluster",
    complaint_col="number_complaints",
    threshold=2,
)
display(complaints_profile)


### 9.4 — Geographic profiling

Geographic variables are used only after clustering, as profiling variables. They are not included in the clustering distance because location could dominate the segmentation and create geographic groups instead of behavioural customer segments.

The scatter plot provides a first visual check of whether any segment is spatially concentrated. Because most customers are located in a dense Lisbon area rectangle, overlap is expected. The purpose of this section is therefore not to define clusters geographically, but to identify whether any segment has a noticeable location pattern that can support interpretation or business actions.


In [ ]:
if {'latitude','longitude'}.issubset(regular.columns):
    plt.figure(figsize=(8, 7))
    sns.scatterplot(data=regular, x='longitude', y='latitude',
                    hue='cluster', palette='tab10', s=8, alpha=0.5, legend='full')
    plt.title('Customer locations by segment (profiling only)')
    plt.show()

In [ ]:
geo_profile = uc.geographic_profile(
    regular,
    label_col="cluster",
)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Geographic interpretation.</b><br><br>
The geographic plot should be interpreted cautiously. A strong overlap between colours means that the final clusters are mainly behavioural rather than geographic, which is desirable because latitude and longitude were intentionally excluded from the clustering distance. If one segment appears more concentrated in a specific area, this can be reported as a profiling insight, but it should not be treated as the main reason for the segment definition.
<br><br>
In practical terms, this analysis can support local marketing decisions, store level planning or regional campaign targeting. However, the final segment names remain based on spending, promotion behaviour, demographic profile and customer stability rather than location alone.
</div>


### 9.5 — Numeric cluster labels

The modelling stage keeps the final groups as numeric labels. Business names are assigned in the characterization stage, after the numerical labels are compared with spend, demographic, loyalty and complaint profiles.

In [ ]:
cluster_labels = pd.DataFrame({
    "cluster": sorted(regular["cluster"].unique())
})
display(cluster_labels)


<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>10) Reattach outliers and export</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">Outliers are not used to fit the KMeans centroids, because their extreme values could distort the segment centres. After the model is fitted on regular customers, the kept aside outliers are projected through the same feature pipeline and assigned to the nearest centroid. This preserves full customer coverage while keeping the learned segments robust.</p>
</div>


In [ ]:
segments = uc.reattach_outliers_and_export(
    regular,
    outliers,
    kmeans,
    distance_cols,
    LOGABS,
    scaler,
    DATA_DIR,
    spend_profile=spend_profile,
    complaints_profile=complaints_profile if "complaints_profile" in locals() else None,
)

In [ ]:
segments.head()

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<code>customer_segments.csv</code> is the required deliverable, containing one final numeric segment assignment per customer. The additional exports support interpretation: <code>segment_summary.csv</code> gives segment sizes, <code>segment_spend_profile.csv</code> supports spend profiling, and <code>segment_complaints_profile.csv</code> highlights service sensitivity. Business names are assigned in the characterization stage.
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>11) Final modelling conclusion</b></h2>
</div>

The final segmentation uses KMeans with `k = 8`, the `spend + promo no groceries` feature set and MinMax scaling.

The choice of `k = 8` is based on the combined evidence from the silhouette grid, UMAP projection, stability analysis and profiling tables. The seven cluster solution was competitive, but it merged customer behaviours that became more meaningful when eight clusters were retained. The selected solution separates promotion oriented customers, loyal grocery customers, vegetarian profiles, wellness shoppers, family shoppers and technology focused customers more clearly.

MinMax scaling was selected because the extreme customers had already been separated before modelling and the remaining variables needed to be compared on a common scale. In this context, MinMax keeps category spend variables and the promotion percentage interpretable within the same distance calculation. Robust scaling was considered, but it compressed the regular customer space less clearly for the final feature set.

Groceries were excluded from the clustering distance because they dominated several candidate solutions and made some segments appear too similar. They remain in the profiling stage, where they are important for understanding customer value and household shopping behaviour. This separation keeps the model focused on differentiating purchasing patterns while still using groceries for interpretation.

The category share, age and children alternative produced fewer negative silhouette values, but the UMAP projection showed weaker visual separation. Other compact feature sets reached higher silhouette values, but they removed too much purchase information and made the final business profiles less useful. The final model is therefore not selected from one metric alone; it is selected as the best compromise between statistical validation, visual separation and segment interpretability.